<a href="https://colab.research.google.com/github/hberkaycoser-hash/HBC/blob/main/E%26PB_SSipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:
import os
import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw, ImageFont
import pandas as pd
import time

# MODEL VE KLASÖR YOLLARI
tflite_path = '/content/drive/MyDrive/colab_notebooks/berkay-bey/2.tflite_3'
image_dir = '/content/drive/MyDrive/colab_notebooks/berkay-bey/123_Street'
output_dir = '/content/drive/MyDrive/colab_notebooks/berkay-bey/123_Street_Outputs'
gt_dir = output_dir  # GT dosyaları çıktı klasöründe

os.makedirs(output_dir, exist_ok=True)

labels = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

def create_cityscapes_label_colormap():
    return np.array([
        [128, 64, 128], [244, 35, 232], [70, 70, 70], [102, 102, 156], [190, 153, 153],
        [153, 153, 153], [250, 170, 30], [220, 220, 0], [107, 142, 35], [152, 251, 152],
        [70, 130, 180], [220, 20, 60], [255, 0, 0], [0, 0, 142], [0, 0, 70],
        [0, 60, 100], [0, 80, 100], [0, 0, 230], [119, 11, 32]
    ], dtype=np.uint8)

colormap = create_cityscapes_label_colormap()

# TensorFlow Lite modelini yükle
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_shape = input_details[0]['shape']
target_h, target_w = input_shape[1], input_shape[2]

def preprocess_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image_resized = image.resize((target_w, target_h), Image.BILINEAR)
    image_np = np.asarray(image_resized).astype(np.float32)
    image_np = image_np / 127.5 - 1.0
    return image, image_np

def predict_segmentation(image_path):
    original_image, input_tensor = preprocess_image(image_path)
    input_tensor = np.expand_dims(input_tensor, axis=0)

    interpreter.set_tensor(input_details[0]['index'], input_tensor)
    interpreter.invoke()

    raw_output = interpreter.get_tensor(output_details[0]['index'])
    logits = raw_output[0]
    seg_map = np.argmax(logits, axis=-1).astype(np.uint8)

    seg_map_img = Image.fromarray(seg_map).resize(original_image.size, Image.NEAREST)
    return original_image, np.array(seg_map_img, dtype=np.uint8)

def visualize_and_save(image_name, image, seg_map):
    seg_color = colormap[seg_map]
    seg_pil = Image.fromarray(seg_color)

    # Grayscale görselleştirme
    max_index = len(labels) - 1
    if max_index > 0:
        seg_vis = (seg_map * (255.0 / max_index)).astype(np.uint8)
    else:
        seg_vis = seg_map.astype(np.uint8)

    # GT ile uyumlu olması için çift "gray" içeren isimle kaydet
    base_name = os.path.splitext(image_name)[0]
    gray_output_path = os.path.join(output_dir, f"{base_name}_gray_gray.png")
    Image.fromarray(seg_vis).save(gray_output_path)

    # Overlay oluştur
    overlay = Image.blend(image.convert('RGBA'), seg_pil.convert('RGBA'), alpha=0.5)

    # Composite görsel boyutları
    width, height = image.width, image.height
    gap = 20
    legend_height = 60
    composite_width = width * 3 + gap * 4
    composite_height = height + legend_height + gap

    composite = Image.new('RGB', (composite_width, composite_height), (255, 255, 255))

    # Görselleri yerleştir
    composite.paste(image, (gap, gap))
    composite.paste(seg_pil, (width + 2 * gap, gap))
    composite.paste(overlay, (2 * width + 3 * gap, gap))

    # Başlık çiz
    draw = ImageDraw.Draw(composite)
    try:
        font_legend = ImageFont.truetype("arial.ttf", 20)
        font_title = ImageFont.truetype("arialbd.ttf", 36)
    except:
        font_legend = ImageFont.load_default()
        font_title = ImageFont.load_default()

    # Başlıklar
    draw.text((gap, 5), "Input image", font=font_title, fill=(0, 0, 0))
    draw.text((width + 2 * gap, 5), "Segmentation map", font=font_title, fill=(0, 0, 0))
    draw.text((2 * width + 3 * gap, 5), "Segmentation overlay", font=font_title, fill=(0, 0, 0))

    # Lejant alanı
    legend_x = gap
    legend_y = height + gap + 10
    row_h = 20
    box_size = 15
    text_x_offset = box_size + 5
    spacing_x = 100

    # Yüzde hesapları
    total_pixels = seg_map.size
    unique, counts = np.unique(seg_map, return_counts=True)
    percentages = {labels[i]: (counts[idx] / total_pixels * 100) for idx, i in enumerate(unique) if i < len(labels)}

    # 1. Satır: İlk 10 etiket
    for idx in range(10):
        x = legend_x + idx * spacing_x
        y = legend_y
        color = tuple(colormap[idx])
        perc = percentages.get(labels[idx], 0.0)
        text = f"{labels[idx]}: {perc:.1f}%"
        draw.rectangle([x, y, x + box_size, y + box_size], fill=color, outline=(0, 0, 0))
        draw.text((x + text_x_offset, y + 1), text, fill=(0, 0, 0), font=font_legend)

    # 2. Satır: Sonraki 9 etiket
    for idx in range(10, 19):
        x = legend_x + (idx - 10) * spacing_x
        y = legend_y + row_h
        color = tuple(colormap[idx])
        perc = percentages.get(labels[idx], 0.0)
        text = f"{labels[idx]}: {perc:.1f}%"
        draw.rectangle([x, y, x + box_size, y + box_size], fill=color, outline=(0, 0, 0))
        draw.text((x + text_x_offset, y + 1), text, fill=(0, 0, 0), font=font_legend)

    # Composite'i kaydet
    composite_output_path = os.path.join(output_dir, f"{base_name}_composite.jpg")
    composite.save(composite_output_path)

    return percentages

def compute_miou(gt_mask, pred_mask, num_classes):
    ious = []
    for cls in range(num_classes):
        gt_cls = (gt_mask == cls)
        pred_cls = (pred_mask == cls)
        intersection = np.logical_and(gt_cls, pred_cls).sum()
        union = np.logical_or(gt_cls, pred_cls).sum()
        ious.append(intersection / union if union > 0 else np.nan)
    return np.nanmean(ious)

# Sonuç listeleri
miou_list = []
results = []
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"Toplam {len(image_files)} resim işlenecek...")
print(f"GT klasörü: {gt_dir}")
print(f"GT klasöründeki dosyalar: {os.listdir(gt_dir)[:5]}")
start_time = time.time()

for idx, image_file in enumerate(image_files):
    print(f"\n⏳ [{idx+1}/{len(image_files)}] İşleniyor: {image_file}")
    img_path = os.path.join(image_dir, image_file)

    try:
        # Segmentasyon tahmini
        original_img, seg_map = predict_segmentation(img_path)

        # Görselleştirme ve kaydetme
        percents = visualize_and_save(image_file, original_img, seg_map)

        # Sonuçları kaydet
        row = {"image": image_file}
        for label in labels:
            row[label] = percents.get(label, 0.0)
        results.append(row)

        # GT dosyası bul ve mIoU hesapla
        base_name = os.path.splitext(image_file)[0]
        gt_file = f"{base_name}_gray_gray.png"  # Çift "gray" formatı

        # GT dosyasını çıktı klasöründe ara
        gt_path = os.path.join(gt_dir, gt_file)

        if os.path.exists(gt_path):
            # GT maskesini uygun boyuta getir
            gt_img = Image.open(gt_path)
            gt_mask = np.array(gt_img)

            # GT maskesini seg_map ile aynı boyuta getir
            if gt_mask.shape != seg_map.shape:
                gt_img = gt_img.resize(original_img.size, Image.NEAREST)
                gt_mask = np.array(gt_img)

            # GT değerlerini ölçeklendir (0-18 aralığına)
            if gt_mask.max() > 18:
                gt_mask = (gt_mask * (18.0 / 255.0)).astype(np.uint8)

            miou = compute_miou(gt_mask, seg_map, len(labels))
            miou_list.append({"image": image_file, "mIoU": miou})
            print(f"   ✅ mIoU hesaplandı: {miou:.4f}")
        else:
            print(f"   ⚠️ GT dosyası bulunamadı: {gt_file}")
            print(f"   🔍 Aranan yol: {gt_path}")

        # İlerleme bilgisi
        elapsed = time.time() - start_time
        avg_time = elapsed / (idx+1)
        remaining = avg_time * (len(image_files) - idx - 1)
        print(f"   ⏱️ Geçen süre: {elapsed:.1f}s, Kalan: {remaining:.1f}s")

    except Exception as e:
        print(f"   ⚠️ Hata: {str(e)}")
        continue

# Excel çıktıları
if results:
    df = pd.DataFrame(results)
    df = df[["image"] + labels]

    # Ortalama satırı ekle
    mean_values = {label: df[label].mean() for label in labels}
    mean_row = {"image": "AVERAGE", **mean_values}
    df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

    excel_output_path = os.path.join(output_dir, '123_Street_segmentation_summary.xlsx')
    df.to_excel(excel_output_path, index=False)
    print(f"\n✅ Segmentasyon raporu kaydedildi: {excel_output_path}")
else:
    print("\n⚠️ Hiçbir resim işlenemedi!")

# ... (Yukarıdaki tüm kodlar aynen kalıyor, son mIoU bölümü yeni kodla değiştirilecek)

# mIoU sonuçları (Her etiket için IoU dahil)
if miou_list:
    df_miou = pd.DataFrame(miou_list)

    print("\n📊 Sınıf başına ortalama IoU analiz ediliyor:")
    all_ious = []
    valid_image_names = []

    for entry in miou_list:
        image_file = entry["image"]
        img_path = os.path.join(image_dir, image_file)
        gt_file = f"{os.path.splitext(image_file)[0]}_gray_gray.png"
        gt_path = os.path.join(gt_dir, gt_file)

        if not os.path.exists(gt_path):
            continue

        original_img, pred_mask = predict_segmentation(img_path)
        gt_mask = np.array(Image.open(gt_path))

        if gt_mask.shape != pred_mask.shape:
            gt_mask = np.array(Image.open(gt_path).resize(original_img.size, Image.NEAREST))

        if gt_mask.max() > len(labels) - 1:
            print(f"⚠️ GT mask max={gt_mask.max()}, dönüşüm uygulanıyor: {image_file}")
            gt_mask = (gt_mask * (18.0 / 255.0)).astype(np.uint8)

        ious = []
        for cls in range(len(labels)):
            gt_cls = (gt_mask == cls)
            pred_cls = (pred_mask == cls)
            intersection = np.logical_and(gt_cls, pred_cls).sum()
            union = np.logical_or(gt_cls, pred_cls).sum()
            iou = intersection / union if union > 0 else np.nan
            ious.append(iou)

        all_ious.append(ious)
        valid_image_names.append(image_file)

    # mIoU dataframe'e her sınıf için ortalama iou ekle
    iou_array = np.array(all_ious)
    mean_ious_per_class = np.nanmean(iou_array, axis=0)

    print("\n✨ Sınıf bazlı mIoU (IoU):")
    per_class_results = {"Class": labels, "IoU": mean_ious_per_class}
    df_class_iou = pd.DataFrame(per_class_results)
    for idx, iou in enumerate(mean_ious_per_class):
        print(f"{labels[idx]:<15}: {iou:.4f}")

    # Ortalama mIoU
    mean_miou = np.nanmean(mean_ious_per_class)
    df_class_iou.loc[len(df_class_iou.index)] = ["AVERAGE", mean_miou]

    # Kaydet
    miou_excel_path = os.path.join(output_dir, '123_Street_mIoU_summary.xlsx')
    df_class_iou.to_excel(miou_excel_path, index=False)

    print(f"\n✅ mIoU raporu kaydedildi: {miou_excel_path}")
    print(f"📌 Ortalama mIoU (per-class average): {mean_miou:.4f}")
else:
    print("⚠️ mIoU hesaplanamadı veya veri bulunamadı")

ValueError: Could not open '/content/drive/MyDrive/colab_notebooks/berkay-bey/2.tflite_3'.